## Question 1

Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between (e.g., prompt_text A_text). What is the exact character length (using Python's len() function, not the number of tokens) of the combined_text string for the row at index 51 (zero-indexed)?

In [11]:
from datasets import load_dataset

# Load train.csv using Hugging Face datasets
dataset = load_dataset(
    "csv",
    data_files="/content/train (1).csv"
)["train"]

# Create the combined_text column
def combine_text(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

dataset = dataset.map(combine_text)

# Find the character length of combined_text for row 51
answer = len(dataset[51]["combined_text"])

print("Answer:", answer)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Answer: 614


## Question 2
Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?  

In [12]:
from transformers import AutoTokenizer

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Print the vocabulary size
print("Vocabulary Size:", tokenizer.vocab_size)

Vocabulary Size: 30522


## Question 3
Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.

In [13]:
from transformers import AutoTokenizer

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Get the ID of the [SEP] token
sep_token_id = tokenizer.sep_token_id

print("SEP Token:", tokenizer.sep_token)
print("SEP Token ID:", sep_token_id)

SEP Token: [SEP]
SEP Token ID: 102


## Question 4
Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors).

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [16]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Convert the Hugging Face Column to a Python list of strings
prompts = list(map(str, dataset["prompt"]))

encoded = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encoded["input_ids"].shape)

torch.Size([2000, 128])


## Question 5
A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer.

In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  

In [17]:
hidden_size = 768
num_attention_heads = 12

head_dimension = hidden_size // num_attention_heads
print(head_dimension)

64


## Question 6
Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object.

What is the exact shape of the last_hidden_state tensor returned?

Note: We follow zero-indexing here.

In [18]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Get the prompt from row 0
text = dataset[0]["prompt"]

# Tokenize using default settings (no manual padding/truncation)
inputs = tokenizer(text, return_tensors="pt")

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)

# Print the shape of the last hidden state
print(outputs.last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])


## Question 7
Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).  

In [19]:
import torch

# Extract the [CLS] embedding (batch 0, token 0)
cls_embedding = outputs.last_hidden_state[0, 0]

# Sum the first 5 values
answer = cls_embedding[:5].sum().item()

# Round to 4 decimal places
print(round(answer, 4))

-1.2001


## Question 8
Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0).

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).  

In [20]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load tokenizer and model with attentions enabled
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

# Input sentence
text = "Light-ion fusion is a technique."

# Tokenize
inputs = tokenizer(text, return_tensors="pt")

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)

# View the tokenization
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(tokens)

# Find the index of "fusion"
fusion_idx = tokens.index("fusion")
print("Fusion token index:", fusion_idx)

# Last layer, first attention head
last_layer_attention = outputs.attentions[-1]      # (batch, heads, seq_len, seq_len)
head0_attention = last_layer_attention[0, 0]       # (seq_len, seq_len)

# Attention from [CLS] (index 0) to "fusion"
attention_weight = head0_attention[0, fusion_idx].item()

print("Attention weight:", round(attention_weight, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Fusion token index: 4
Attention weight: 0.1025


## Question 9
Context-Aware Embeddings
Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.

In [21]:
from sentence_transformers import SentenceTransformer, util

# Load the model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Get prompt and Option B from row 0
prompt = dataset[0]["prompt"]
option_b = dataset[0]["B"]

# Generate embeddings
prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_b_embedding = model.encode(option_b, convert_to_tensor=True)

# Cosine similarity
similarity = util.cos_sim(prompt_embedding, option_b_embedding)

print("Cosine Similarity:", round(similarity.item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Cosine Similarity: 0.7658


## Question 10
Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set?

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
import numpy as np
from tqdm import tqdm

# Load MiniLM
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

options = ["A", "B", "C", "D", "E"]

tfidf_top3 = []
minilm_top3 = []

#------------------------------------------------------
# TF-IDF PIPELINE
#------------------------------------------------------

for row in tqdm(dataset, desc="TF-IDF"):
    texts = [row["prompt"]] + [row[o] for o in options]

    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(texts)

    sims = cosine_similarity(X[0:1], X[1:]).flatten()

    ranking = np.argsort(sims)[::-1]
    tfidf_top3.append([options[i] for i in ranking[:3]])

#------------------------------------------------------
# MiniLM PIPELINE
#------------------------------------------------------

for row in tqdm(dataset, desc="MiniLM"):
    prompt_embedding = model.encode(
        row["prompt"],
        convert_to_tensor=True
    )

    option_embeddings = model.encode(
        [row[o] for o in options],
        convert_to_tensor=True
    )

    sims = util.cos_sim(prompt_embedding, option_embeddings)[0].cpu().numpy()

    ranking = np.argsort(sims)[::-1]
    minilm_top3.append([options[i] for i in ranking[:3]])

#------------------------------------------------------
# MAP@3
#------------------------------------------------------

def apk(actual, predicted, k=3):
    predicted = predicted[:k]

    for i, p in enumerate(predicted):
        if p == actual:
            return 1.0 / (i + 1)

    return 0.0

scores = [
    apk(row["answer"], pred, 3)
    for row, pred in zip(dataset, minilm_top3)
]

map3 = np.mean(scores)

#------------------------------------------------------
# Improvement Count
#------------------------------------------------------

count = 0

for row, tfidf_pred, minilm_pred in zip(dataset, tfidf_top3, minilm_top3):
    ans = row["answer"]

    if ans not in tfidf_pred and ans in minilm_pred:
        count += 1

print("MiniLM MAP@3:", round(map3, 6))
print("Recovered by MiniLM:", count)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

MiniLM:  55%|█████▌    | 1104/2000 [04:08<07:39,  1.95it/s]

## Question 11
Zero-shot classification concepts
Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [ ]:
from transformers import pipeline

# Initialize zero-shot classification pipeline
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# Get the prompt from row index 1
sequence = dataset[1]["prompt"]

# Candidate labels: Options A, B, and C
candidate_labels = [
    dataset[1]["A"],
    dataset[1]["B"],
    dataset[1]["C"]
]

# Run zero-shot classification
result = classifier(sequence, candidate_labels)

# Display results
print("Top Label:", result["labels"][0])
print("Top Score:", round(result["scores"][0], 4))

## Question 12
Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True.

What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [ ]:
from transformers import pipeline

# Initialize zero-shot classification pipeline
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# Get the prompt from row index 1
sequence = dataset[1]["prompt"]

# Candidate labels: Options A, B, and C
candidate_labels = [
    dataset[1]["A"],
    dataset[1]["B"],
    dataset[1]["C"]
]

# Run zero-shot classification
result = classifier(sequence, candidate_labels)

# Display results
print("Top Label:", result["labels"][0])
print("Top Score:", round(result["scores"][0], 4))

## Question 13
Let's try Generative AI instead of Classification.

Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B."
Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model?

In [ ]:
from transformers import pipeline

# Load FLAN-T5 Small
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)

# Construct the prompt for row 0
text = (
    f"Question: {dataset[0]['prompt']}. "
    f"Is the correct answer A: {dataset[0]['A']} "
    f"or B: {dataset[0]['B']}? "
    f"Answer with just the letter A or B."
)

# Generate the answer
result = generator(text, max_new_tokens=5)

# Print the exact output string
print(result[0]["generated_text"])